# Evaluate averaged training curves by seed

Notebook này đọc các file `outputs/runs/*.json`, group theo `Model`, `Optimizer`, `Alpha`, lấy trung bình từng epoch qua seed `0, 1, 2`, rồi vẽ `train loss`, `train accuracy`, `val loss`, `val accuracy`.


In [ ]:
from collections import defaultdict
from pathlib import Path
import json

import matplotlib.pyplot as plt

RUN_DIR = Path("outputs/runs")
FIG_DIR = Path("outputs")
REQUIRED_SEEDS = {0, 1, 2}
INCLUDE_INCOMPLETE = False  # chỉ vẽ nhóm có đủ seed 0, 1, 2

METRICS = {
    "train_loss": "Train loss",
    "train_acc": "Train accuracy",
    "val_loss": "Val loss",
    "val_acc": "Val accuracy",
}


In [ ]:
def load_runs(run_dir=RUN_DIR):
    runs = []
    for run_file in sorted(run_dir.glob("*.json")):
        with run_file.open("r", encoding="utf-8") as f:
            run = json.load(f)
        run["_file"] = run_file.name
        runs.append(run)
    return runs


def group_runs_by_config(runs):
    groups = defaultdict(list)
    for run in runs:
        key = (run["Model"], run["Optimizer"], float(run["Alpha"]))
        groups[key].append(run)
    return dict(groups)


def mean_curve(curves):
    min_len = min(len(curve) for curve in curves)
    return [
        sum(float(curve[idx]) for curve in curves) / len(curves)
        for idx in range(min_len)
    ]


def average_group(runs):
    averaged = {}
    for output_key, json_key in METRICS.items():
        curves = [run[json_key] for run in runs]
        averaged[output_key] = mean_curve(curves)
    return averaged


def build_averaged_histories(required_seeds=REQUIRED_SEEDS, include_incomplete=INCLUDE_INCOMPLETE):
    runs = load_runs()
    groups = group_runs_by_config(runs)

    histories = {}
    skipped = []
    for (model, optimizer, alpha), group in sorted(groups.items()):
        by_seed = {int(run["Seed"]): run for run in group}
        seeds = sorted(by_seed)
        missing = sorted(required_seeds - set(seeds))

        if missing and not include_incomplete:
            skipped.append({
                "label": f"{model} | {optimizer} | alpha={alpha:g}",
                "available_seeds": seeds,
                "missing_seeds": missing,
            })
            continue

        selected_seeds = sorted(required_seeds & set(seeds)) if not include_incomplete else seeds
        selected_runs = [by_seed[seed] for seed in selected_seeds]
        label = f"{optimizer} alpha={alpha:g}"
        histories[label] = average_group(selected_runs)
        histories[label]["seeds"] = selected_seeds
        histories[label]["model"] = model
        histories[label]["optimizer"] = optimizer
        histories[label]["alpha"] = alpha
        histories[label]["plot_label"] = f"alpha={alpha:g}"

    return histories, skipped


In [ ]:
histories, skipped = build_averaged_histories()

print(f"Loaded averaged groups: {len(histories)}")
for label, history in histories.items():
    print(
        f"{label:24s} | seeds={history['seeds']} | "
        f"epochs={len(history['val_acc'])} | "
        f"best avg val acc={max(history['val_acc']):.4f}"
    )

if skipped:
    print("\nSkipped groups missing required seeds:")
    for item in skipped:
        print(
            f"{item['label']} | available={item['available_seeds']} | "
            f"missing={item['missing_seeds']}"
        )


In [ ]:
def split_histories_by_optimizer(histories):
    by_optimizer = defaultdict(dict)
    for label, history in histories.items():
        optimizer = history["optimizer"]
        by_optimizer[optimizer][history["plot_label"]] = history
    return dict(by_optimizer)


def safe_filename(name):
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in name)


def plot_optimizer_history(optimizer, optimizer_histories, fig_dir=FIG_DIR):
    if not optimizer_histories:
        raise ValueError("No averaged histories to plot. Check RUN_DIR and REQUIRED_SEEDS.")

    fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
    plot_specs = [
        ("train_loss", "Train Loss", "Loss", axes[0, 0]),
        ("train_acc", "Train Accuracy", "Accuracy (%)", axes[0, 1]),
        ("val_loss", "Validation Loss", "Loss", axes[1, 0]),
        ("val_acc", "Validation Accuracy", "Accuracy (%)", axes[1, 1]),
    ]

    for metric_key, title, ylabel, ax in plot_specs:
        for label, history in optimizer_histories.items():
            epochs = range(1, len(history[metric_key]) + 1)
            ax.plot(epochs, history[metric_key], linewidth=1.8, label=label)
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5))
    used_seed_text = ", ".join(
        f"{label}: {history['seeds']}" for label, history in optimizer_histories.items()
    )
    fig.suptitle(f"{optimizer} average training curves by seed", y=1.02)
    fig.text(0.5, 0.005, f"Averaged seeds - {used_seed_text}", ha="center")
    fig.tight_layout()

    fig_dir.mkdir(parents=True, exist_ok=True)
    fig_path = fig_dir / f"avg_seed_curves_{safe_filename(optimizer)}.png"
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    return fig_path


saved_figures = []
for optimizer, optimizer_histories in split_histories_by_optimizer(histories).items():
    saved_figures.append(plot_optimizer_history(optimizer, optimizer_histories))

for fig_path in saved_figures:
    print(f"Saved figure: {fig_path}")
